In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('data/simulated_data_formatted_alpha_rnd_1.txt',
                 delimiter=' ', index_col=0)

# number of households stays constant
# number of individuals might be different (households are sampled, and can be sampled multiple times)
# individuals do not have an order, they are ordered the same way they were entered in the study
# estimating 12 parameters (which are dependent on covariates)

In [3]:
df.drop(columns=['variant', 'id_simu', 'select_process'], inplace=True)

In [4]:
df.head()

,id_hh,hh_size,date_sympt,infect_status,end_followup,age,protected
id_patient,,,,,,,
95,24-a-r1,5,38,1,129,2,0
96,24-a-r1,5,37,1,129,2,0
97,24-a-r1,5,33,1,129,0,0
98,24-a-r1,5,44,1,129,0,0
99,24-a-r1,5,36,1,129,0,0


In [5]:
df['id_hh'].nunique()

128

In [6]:
# hh_size is always the same for each household
df[['hh_size']].value_counts()

hh_size
4          231
5          194
3           41
6           40
2            8
Name: count, dtype: int64

In [7]:
np.max(df.loc[df['date_sympt']<1000, 'date_sympt']) #.value_counts()

123

In [8]:
df['infect_status'].value_counts()

infect_status
1    221
0    221
2     72
Name: count, dtype: int64

In [9]:
# end_followup is always the same for each household
df[['id_hh', 'end_followup']].value_counts()

id_hh      end_followup
4554-a-r1  112             6
1866-a-r1  109             6
4938-a-r1  109             6
1738-a-r1  108             6
2717-a-r1  129             6
                          ..
5819-a-r1  99              2
6234-a-r1  98              2
2079-a-r1  154             2
1515-a-r1  79              2
4587-a-r1  77              2
Name: count, Length: 128, dtype: int64

In [10]:
df['age'].value_counts()

age
2    349
1    102
0     63
Name: count, dtype: int64

In [11]:
df['protected'].value_counts()

protected
0    484
1     30
Name: count, dtype: int64

# Convert data to long format

the columns are individuals, followed by the date of the end of follow-up,
each row defines are different variable
- date of symptoms (normalized to 0-1 by dividing by the maximum date)
- household id (normalized to 0-1 by dividing by the number of households)
- infection status (-1: not infected, 0: infected+symptomatic, 1: infected+asymptomatic)
- age groups (-1: <6 years old, 0: 6-11 years old, 1: >11 years old)
- protected (0: not protected, 1: protected)

In [12]:
from helper_functions import normalize_household_data

In [19]:
household_data = normalize_household_data(df, minimal_length=8)
household_data

array([[[ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
        [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
        [ 0.033,  1.   ,  0.   ,  0.   ,  0.   ,  0.   ],
        ...,
        [ 0.038,  1.   ,  0.   ,  0.   ,  1.   ,  0.   ],
        [ 0.044,  1.   ,  0.   ,  0.   ,  0.   ,  0.   ],
        [ 0.129, -1.   , -1.   , -1.   , -1.   , -1.   ]],

       [[ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
        [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
        [ 0.036,  1.   ,  0.   ,  0.   ,  1.   ,  0.   ],
        ...,
        [ 1.   ,  0.   ,  0.   ,  0.   ,  1.   ,  0.   ],
        [ 1.   ,  0.   ,  0.   ,  0.   ,  1.   ,  1.   ],
        [ 1.   ,  0.   ,  0.   ,  0.   ,  1.   ,  0.   ]],

       [[ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
        [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
        [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
        ...,
        [ 0.101, -1.   , -1.   , -1.   , -1.   , -1.   ],
        [ 1.   ,  0.   ,  0. 

In [20]:
2** int(np.ceil(np.log2(household_data.shape[1]))), household_data.shape[1]

(8, 8)

In [21]:
household_data[-1]

array([[ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
       [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
       [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
       [ 0.034,  1.   ,  0.   ,  0.   ,  1.   ,  0.   ],
       [ 0.034,  1.   ,  0.   ,  0.   ,  1.   ,  0.   ],
       [ 0.08 , -1.   , -1.   , -1.   , -1.   , -1.   ],
       [ 1.   ,  0.   ,  0.   ,  0.   ,  1.   ,  0.   ],
       [ 1.   ,  0.   ,  0.   ,  0.   ,  1.   ,  0.   ]])

In [22]:
negative_one_indices = np.argmax(household_data == -1, axis=0)

In [23]:
household_data.shape

(128, 8, 6)

In [24]:
negative_one_indices

array([[ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0, 30, 30, 30, 30, 30],
       [ 0,  9,  9,  9,  9,  9],
       [ 0,  1,  1,  1,  1,  1],
       [ 0,  2,  2,  2,  2,  2],
       [ 0,  3,  3,  3,  3,  3],
       [ 0,  0,  0,  0,  0,  0]])